In [ ]:
# From the Inference ipynb

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

rng = np.random.default_rng(100)

def purchase_probability(price):
    price = np.asarray(price, dtype=float)
    return (
        0.55 * np.exp(-np.logaddexp(0.0, (price - 13.9) / 0.8))
        + 0.30 * np.exp(-np.logaddexp(0.0, (price - 30.9) / 1.2))
        + 0.15 * np.exp(-np.logaddexp(0.0, (price - 58.4) / 1.8))
    )

def sales(price=14.0, rng=None, mean_customers=30):
    price = float(price)
    if not np.isfinite(price):
        raise ValueError("price must be finite")
    if price < 0:
        raise ValueError("price must be nonnegative")
    mean_customers = float(mean_customers)
    if not np.isfinite(mean_customers) or mean_customers < 0:
        raise ValueError("mean_customers must be finite and nonnegative")
    rng = np.random.default_rng(rng)
    customer_count = rng.poisson(mean_customers)
    probability = purchase_probability(price)
    return rng.binomial(1, probability, size=customer_count).astype(np.int8)

In [ ]:
def estimate_gradient(price, *, step=1.0, mean_customers=300, rng=None):
    if step <= 0:
        raise ValueError("step must be positive")
    rng = np.random.default_rng(rng)
    price = float(np.clip(price, 0, 75))
    price_low = max(0, price - step)
    price_high = min(75, price + step)
    low = price_low * sales(price_low, mean_customers=mean_customers, rng=rng)
    high = price_high * sales(price_high, mean_customers=mean_customers, rng=rng)
    if min(len(low), len(high)) < 2:
        raise ValueError("Need at least two customers per arm; increase mean_customers.")

    difference = high.mean() - low.mean()
    se = np.sqrt(low.var(ddof=1) / len(low) + high.var(ddof=1) / len(high))
    gap = price_high - price_low
    return {
        "price": price,
        "gradient": difference / gap,
        "gradient_se": se / gap,
        "ci_low": (difference - 1.645 * se) / gap,
        "ci_high": (difference + 1.645 * se) / gap,
        "customers": len(low) + len(high),
        "revenue": low.sum() + high.sum(),  
    }

def gradient_ascent(start_price, iterations=80, step=1.0,
                    learning_rate=2.0, mean_customers=300, seed=100):
    if iterations < 1:
        raise ValueError("iterations must be at least 1")
    search_rng = np.random.default_rng(seed)
    price = float(np.clip(start_price, 0, 75))
    records = []
    for iteration in range(iterations):
        estimate = estimate_gradient(price, step=step,
                                     mean_customers=mean_customers, rng=search_rng)
        rate = learning_rate / np.sqrt(iteration + 1)
        move = np.clip(rate * estimate["gradient"], -2.0, 2.0)
        next_price = float(np.clip(price + move, 0, 75))
        records.append({"iteration": iteration + 1, **estimate,
                        "learning_rate": rate, "next_price": next_price})
        price = next_price
    return pd.DataFrame(records)

# ---------------------------------------------------------------------------
# Step 1: grid search over (0, 70], tracking customers and revenue
# ---------------------------------------------------------------------------
total_customers = 0        # every customer we experiment on
total_revenue = 0.0        # revenue earned from those customers
OPTIMAL = 11.925           # expected revenue per customer at the global optimum

grid_prices = np.arange(2, 70.01, 2.0)
grid_rows = []
for p in grid_prices:
    r = sales(p, mean_customers=100, rng=rng) * p      # revenue per arriving customer
    total_customers += len(r)
    total_revenue += r.sum()
    grid_rows.append({"price": p, "mean_rev": r.mean(),
                      "se": r.std(ddof=1) / np.sqrt(len(r)), "customers": len(r)})
grid = pd.DataFrame(grid_rows)
print(grid.round(2).to_string(index=False))

# ---------------------------------------------------------------------------
# Step 2: starting points = two best grid peaks that are at least $8 apart
# ---------------------------------------------------------------------------
m = grid.mean_rev.to_numpy()
padded = np.r_[-np.inf, m, -np.inf]
is_peak = (m > padded[:-2]) & (m >= padded[2:])
peaks = grid[is_peak].sort_values("mean_rev", ascending=False)

starts = []
for p in peaks.price:
    if all(abs(p - q) >= 8 for q in starts):
        starts.append(p)
    if len(starts) == 2:
        break
print("\nGradient ascent starting prices:", starts)

# ---------------------------------------------------------------------------
# Step 3: local search (gradient ascent) from each start
# ---------------------------------------------------------------------------
paths, finals = {}, []
for i, s in enumerate(starts):
    path = gradient_ascent(s, iterations=30, mean_customers=200, seed=1000 + i)
    paths[s] = path
    total_customers += path.customers.sum()
    total_revenue += path.revenue.sum()
    finals.append(path.next_price.iloc[-1])
    print(f"start {s:g} -> final {finals[-1]:.2f} ({path.customers.sum():,} customers)")

# ---------------------------------------------------------------------------
# Step 4: compare the finalists with a two-sample Z-test on fresh customers
# ---------------------------------------------------------------------------
final_data = {}
for f in finals:
    r = sales(f, mean_customers=3000, rng=rng) * f
    total_customers += len(r)
    total_revenue += r.sum()
    final_data[f] = r
    print(f"price {f:.2f}: mean revenue/customer {r.mean():.3f} (n={len(r)})")

best_price = max(final_data, key=lambda f: final_data[f].mean())
other = [f for f in finals if f != best_price][0]
A, B = final_data[best_price], final_data[other]
diff = A.mean() - B.mean()
se = np.sqrt(A.var(ddof=1) / len(A) + B.var(ddof=1) / len(B))
print(f"\nDifference (best - other): {diff:.3f}, "
      f"95% CI ({diff - 1.96 * se:.3f}, {diff + 1.96 * se:.3f})")
print("Reject H0 of equal revenue" if abs(diff / se) > 1.96
      else "Cannot distinguish the two prices")

# ---------------------------------------------------------------------------
# Results for the write-up
# ---------------------------------------------------------------------------
print(f"\nChosen price:               ${best_price:.2f}")
print(f"Customers experimented on:  {total_customers:,}")
print(f"Revenue during experiments: ${total_revenue:,.0f}")
print(f"Revenue per customer:       ${total_revenue / total_customers:.3f}")
print(f"% of optimal captured:      {total_revenue / (total_customers * OPTIMAL):.1%}")

# ---------------------------------------------------------------------------
# Plots for the write-up
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].errorbar(grid.price, grid.mean_rev, yerr=1.645 * grid.se, fmt="o", ms=3, capsize=2)
ax[0].set(xlabel="Price", ylabel="Mean revenue / customer", title="Grid search (90% CIs)")
for s, path in paths.items():
    ax[1].plot(np.r_[s, path.next_price.to_numpy()], label=f"start ${s:g}")
ax[1].set(xlabel="Iteration", ylabel="Price", title="Gradient ascent paths")
ax[1].legend()
plt.tight_layout()
plt.show()
